# Multiple data sets
The same sample is often measured more than once, for example with two incident energies on the same instrument. The data sets then have different resolutions, energy windows and intensity scales, and some features of the sample are only visible in one of them.

Here we fit two such data sets together. The sample has three Lorentzians: `L1` is narrow and only resolved in the fine-resolution data; `L2` is seen in both; `L3` is broad and only fits inside the energy window of the coarse-resolution data. We share `L1` and `L2` between the two fits, so their parameters are determined by both data sets at once.

In [ ]:
# Imports
import pooch

import easydynamics as edyn

# Make the plots interactive
%matplotlib widget

We load two artificial data sets of the same sample. The fine setting has a Gaussian resolution of width 0.01 meV and an energy window of ±1 meV. The coarse setting has a resolution of width 0.1 meV with Lorentzian tails and a window of ±16 meV, and its intensities are on another scale than those of the fine setting. The two `Q` grids overlap, but each has one `Q` value the other does not.

In the sample, the width of `L2` grows as $Q^2$, like Brownian diffusion.

In [ ]:
fine_experiment = edyn.Experiment(display_name='Fine resolution')
file_path = pooch.retrieve(
    url='https://github.com/easyscience/dynamics-lib/raw/refs/heads/joint-analysis-tutorial/docs/docs/tutorials/data/fake_fine_resolution_data.hdf5',
    known_hash='05c708719e3325bed7c4b417fd7c9495debe1074cf3fbc8fb2ce8efc466cd912',
)
fine_experiment.load_hdf5(filename=file_path)

coarse_experiment = edyn.Experiment(display_name='Coarse resolution')
file_path = pooch.retrieve(
    url='https://github.com/easyscience/dynamics-lib/raw/refs/heads/joint-analysis-tutorial/docs/docs/tutorials/data/fake_coarse_resolution_data.hdf5',
    known_hash='48f2d537eb3ff9347299b20a09e2cecc8d089d0fee4deab26b5b17122427b160',
)
coarse_experiment.load_hdf5(filename=file_path)

In [ ]:
fine_experiment.plot_data(slicer=True)

In [ ]:
coarse_experiment.plot_data(slicer=True)

We create the three Lorentzians once and build a `SampleModel` for each data set from them. Appending the same object to both models is what marks it as shared, so `L1` and `L2` are shared. In the coarse data `L1` is much narrower than the resolution and cannot be resolved; because it is shared, its width comes from the fine data. `L3` is so broad that in the fine data it is a flat background, which the `BackgroundModel` absorbs, so it is only in the coarse model.

In [ ]:
L1 = edyn.Lorentzian(name='L1', area=0.3, width=0.01)
L2 = edyn.Lorentzian(name='L2', area=1.0, width=0.1)
L3 = edyn.Lorentzian(name='L3', area=2.0, width=3.0)

fine_model = edyn.SampleModel(components=edyn.ComponentCollection(components=[L1, L2]))
coarse_model = edyn.SampleModel(components=edyn.ComponentCollection(components=[L1, L2, L3]))

Each data set has its own resolution and background, so each gets its own `InstrumentModel`. We take the resolutions as known and fix them. The fine resolution is a Gaussian.

In [ ]:
fine_resolution_model = edyn.ResolutionModel(components=edyn.Gaussian(width=0.01))
fine_resolution_model.fix_all_parameters()

fine_instrument_model = edyn.InstrumentModel(
    resolution_model=fine_resolution_model,
    background_model=edyn.BackgroundModel(components=edyn.Polynomial(coefficients=[0.02])),
)

fine_analysis = edyn.Analysis(
    display_name='Fine resolution',
    experiment=fine_experiment,
    sample_model=fine_model,
    instrument_model=fine_instrument_model,
)

The coarse resolution is a broader Gaussian with Lorentzian tails. The areas of the two components add up to 1.

In [ ]:
coarse_resolution_model = edyn.ResolutionModel(
    components=edyn.ComponentCollection(
        components=[
            edyn.Gaussian(name='Resolution Gaussian', area=0.9, width=0.1),
            edyn.Lorentzian(name='Resolution tails', area=0.1, width=0.1),
        ]
    )
)
coarse_resolution_model.fix_all_parameters()

coarse_instrument_model = edyn.InstrumentModel(
    resolution_model=coarse_resolution_model,
    background_model=edyn.BackgroundModel(components=edyn.Polynomial(coefficients=[0.1])),
)

coarse_analysis = edyn.Analysis(
    display_name='Coarse resolution',
    experiment=coarse_experiment,
    sample_model=coarse_model,
    instrument_model=coarse_instrument_model,
)

Two measurements are rarely on the same intensity scale. Each `Analysis` has a `scale` that its whole model, sample and background, is multiplied by. It is fixed at 1 by default. We keep the fine data as the reference and free the scale of the coarse analysis.

In [ ]:
coarse_analysis.scale.fixed = False

As always, we check the start guesses before fitting.

In [ ]:
fine_analysis.plot_data_and_model()

In [ ]:
coarse_analysis.plot_data_and_model()

A `JointAnalysis` fits several analyses together. At every `Q` value the two analyses have in common, the `L1` and `L2` parameters of the coarse analysis follow those of the fine analysis, so they are fitted once against both data sets. At the one `Q` value only the coarse data has, the coarse analysis keeps its own copies. Components that appear in only one model are fitted as usual.

In [ ]:
joint_analysis = edyn.JointAnalysis(analyses=[fine_analysis, coarse_analysis])
print(joint_analysis)

`fit` fits all `Q` of both data sets simultaneously.

In [ ]:
joint_analysis.fit()
fine_analysis.plot_data_and_model()

In [ ]:
coarse_analysis.plot_data_and_model()

The scales of the two analyses. The coarse data were generated 3.2 times more intense than the fine data.

In [ ]:
print(joint_analysis.scales)

Where the `Q` values agree, the shared parameters belong to the first analysis in the list, and the copies in the other analyses follow them. `parameters_to_dataset` on the `JointAnalysis` collects the parameters of the shared components over the `Q` values of all the data sets: eight from each, of which seven are in common. At the one `Q` value only the coarse data has (2.4 Å⁻¹), `L1` cannot be resolved and nothing ties it to the fine data, so the parameters in that row are poorly determined.

In [ ]:
joint_analysis.parameters_to_dataset()

The dataset can be used like the one from `Analysis.parameters_to_dataset`. As in Tutorial 1, we fit the `L2` width to a Brownian diffusion model with a `ParameterAnalysis`.

In [ ]:
brownian = edyn.BrownianTranslationalDiffusion(
    name='Brownian diffusion', lorentzian_name='L2', diffusion_coefficient=1e-9
)
binding = edyn.FitBinding(model=brownian, targets=['width'])

parameter_analysis = edyn.ParameterAnalysis(
    parameters=joint_analysis.parameters_to_dataset(), bindings=binding
)
parameter_analysis.fit()
parameter_analysis.plot()

The fitted diffusion coefficient. The data were generated with a width of $0.08\,\mathrm{meV\,Å^2}\cdot Q^2$, which corresponds to $D = 0.08\,\mathrm{meV\,Å^2}/\hbar \approx 1.2 \times 10^{-9}\,\mathrm{m^2/s}$.

In [ ]:
brownian.get_global_variables()

## Sharing a diffusion model
Now that we know `L2` follows Brownian diffusion, we can fit the diffusion model directly to both data sets, as in Tutorial 1. A diffusion model object used in several sample models is shared too: its `Q` grid becomes the union of their `Q` values, each sample model uses the components at its own `Q` values, and the diffusion coefficient is fitted against all the data. The new sample models start from the original guesses for `L1` and `L3`, and we reuse the instrument models, which keep their fitted backgrounds and the free scale. Create all the sample models before fitting: a new one can extend the grid, which regenerates the diffusion model's per-Q components and resets any per-Q parameters it has.

In [ ]:
brownian = edyn.BrownianTranslationalDiffusion(
    name='Brownian diffusion', lorentzian_name='L2', diffusion_coefficient=1e-9
)
fine_model = edyn.SampleModel(components=L1, diffusion_models=brownian)
coarse_model = edyn.SampleModel(
    components=edyn.ComponentCollection(components=[L1, L3]), diffusion_models=brownian
)

fine_analysis = edyn.Analysis(
    display_name='Fine resolution',
    experiment=fine_experiment,
    sample_model=fine_model,
    instrument_model=fine_instrument_model,
)
coarse_analysis = edyn.Analysis(
    display_name='Coarse resolution',
    experiment=coarse_experiment,
    sample_model=coarse_model,
    instrument_model=coarse_instrument_model,
)

joint_analysis = edyn.JointAnalysis(analyses=[fine_analysis, coarse_analysis])
joint_analysis.fit()
fine_analysis.plot_data_and_model()

In [ ]:
brownian.get_global_variables()

Components are shared where the `Q` values of the data sets agree within the `Q_tolerance` of the `JointAnalysis`, which is 0.01 Å⁻¹ unless you set another, e.g. `edyn.JointAnalysis(analyses=[...], Q_tolerance=0.05)`. If no `Q` values agree, nothing is shared and a warning is given; rebin the data onto a common `Q` grid or increase `Q_tolerance` to share components. Parameters that do not depend on `Q`, such as a diffusion coefficient, are not tied to the `Q` grid: a diffusion model object used in several sample models is shared across all of them, whatever their `Q` values.